# Regresión, Sesgo-Varianza, Selección de Modelos II

Se van a comparar modelos regularizados (Ridge y Lasso) frente a la práctica anterior (sin regularización),
analizando MSE de entrenamiento y validación mediante validación cruzada 10-Fold.
Se prueban grados 1..15 y alphas (lambda) en (0.01, 0.1, 1, 10, 100).

In [ ]:
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge, Lasso
from sklearn.model_selection import KFold, cross_validate
from sklearn.exceptions import ConvergenceWarning

import matplotlib.pyplot as plt
import warnings

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)
warnings.filterwarnings("always", category=ConvergenceWarning)


In [ ]:
dataset_files = [
    "dataset_1.csv","dataset_2.csv","dataset_3.csv",
    "dataset_4.csv","dataset_5.csv","dataset_6.csv"
]
DEGREES = list(range(1, 16))
ALPHAS = [0.01, 0.1, 1, 10, 100]
K_FOLDS = 10
SEED = 11


In [ ]:
# Grados óptimos OLS de la práctica anterior
ols_best_degree = {
    "dataset_1.csv": 5,
    "dataset_2.csv": 5,
    "dataset_3.csv": 5,
    "dataset_4.csv": 9,
    "dataset_5.csv": 2,
    "dataset_6.csv": 7,
}

In [ ]:
def infer_xy_columns(df: pd.DataFrame):
    cols = list(df.columns)
    if "x" in cols and "y" in cols:
        return "x", "y"
    if df.shape[1] == 2:
        return cols[0], cols[1]
    num_cols = [c for c in cols if pd.api.types.is_numeric_dtype(df[c])]
    if len(num_cols) >= 2:
        return num_cols[0], num_cols[-1]
    return cols[0], cols[-1]

def build_pipe(model_type: str, degree: int, alpha: float, max_iter_lasso: int = 30000):
    if model_type == "ridge":
        reg = Ridge(alpha=alpha)
    elif model_type == "lasso":
        reg = Lasso(alpha=alpha, max_iter=max_iter_lasso)
    else:
        raise ValueError("model_type debe ser 'ridge' o 'lasso'")

    return Pipeline([
        ("poly", PolynomialFeatures(degree=degree, include_bias=False)),
        ("reg", reg)
    ])

def cv_mse_train_val(model, X, y, k=10, seed=11):
    cv = KFold(n_splits=k, shuffle=True, random_state=seed)
    res = cross_validate(
        model, X, y,
        cv=cv,
        scoring="neg_mean_squared_error",
        return_train_score=True,
        error_score=np.nan
    )
    train_mse = -res["train_score"]
    val_mse = -res["test_score"]
    return float(np.nanmean(train_mse)), float(np.nanmean(val_mse)), int(np.sum(~np.isnan(val_mse)))

def grid_results(model_type: str, X, y, degrees, alphas):
    rows = []
    for deg in degrees:
        for a in alphas:
            model = build_pipe(model_type, deg, a)
            with warnings.catch_warnings(record=True) as w:
                warnings.simplefilter("always", ConvergenceWarning)
                mse_tr, mse_va, ok = cv_mse_train_val(model, X, y, k=K_FOLDS, seed=SEED)
                conv_warn = sum(1 for ww in w if issubclass(ww.category, ConvergenceWarning))

            rows.append({
                "model": model_type,
                "degree": deg,
                "alpha": a,
                "mse_train": mse_tr,
                "mse_val": mse_va,
                "folds_ok": ok,
                "conv_warnings": conv_warn
            })
    return pd.DataFrame(rows)

def pick_best(df_grid: pd.DataFrame):
    df_sorted = df_grid.sort_values(["mse_val", "degree", "alpha"], ascending=[True, True, True]).reset_index(drop=True)
    return df_sorted.iloc[0]


In [ ]:
all_best = []

for fname in dataset_files:
    print("="*90)
    print("DATASET:", fname)
    print("="*90)

    df = pd.read_csv("../data/" + fname).dropna().copy()
    x_col, y_col = infer_xy_columns(df)

    X = df[[x_col]].to_numpy()
    y = df[y_col].to_numpy()

    # Exploratorio mínimo
    display(df.head())
    display(df.describe())

    plt.figure()
    plt.scatter(X[:,0], y, s=15)
    plt.title(fname + " scatter")
    plt.xlabel(x_col); plt.ylabel(y_col)
    plt.show()

    # Ridge y Lasso
    df_ridge = grid_results("ridge", X, y, DEGREES, ALPHAS)
    df_lasso = grid_results("lasso", X, y, DEGREES, ALPHAS)

    best_ridge = pick_best(df_ridge)
    best_lasso = pick_best(df_lasso)

    print("Mejor Ridge:", dict(best_ridge))
    print("Mejor Lasso:", dict(best_lasso))

    all_best.append({
        "dataset": fname,
        "ols_degree_prev": ols_best_degree.get(fname, np.nan),

        "ridge_degree": int(best_ridge["degree"]),
        "ridge_alpha": float(best_ridge["alpha"]),
        "ridge_mse_train": float(best_ridge["mse_train"]),
        "ridge_mse_val": float(best_ridge["mse_val"]),

        "lasso_degree": int(best_lasso["degree"]),
        "lasso_alpha": float(best_lasso["alpha"]),
        "lasso_mse_train": float(best_lasso["mse_train"]),
        "lasso_mse_val": float(best_lasso["mse_val"]),
        "lasso_conv_warnings": int(best_lasso["conv_warnings"]),
        "lasso_folds_ok": int(best_lasso["folds_ok"])
    })

    # Gráfica 1: Ridge MSE val vs grado por alpha
    plt.figure()
    for a in ALPHAS:
        tmp = df_ridge[df_ridge["alpha"] == a].sort_values("degree")
        plt.plot(tmp["degree"], tmp["mse_val"], marker="o", label=f"a={a}")
    plt.title(fname + " Ridge: MSE val vs grado")
    plt.xlabel("grado"); plt.ylabel("MSE val")
    plt.legend()
    plt.show()

    # Gráfica 2: Ridge train vs val para alpha óptimo
    a_star = float(best_ridge["alpha"])
    tmp = df_ridge[df_ridge["alpha"] == a_star].sort_values("degree")
    plt.figure()
    plt.plot(tmp["degree"], tmp["mse_train"], marker="o", label="train")
    plt.plot(tmp["degree"], tmp["mse_val"], marker="o", label="val")
    plt.title(fname + f" Ridge (alpha={a_star}): train vs val")
    plt.xlabel("grado"); plt.ylabel("MSE")
    plt.legend()
    plt.show()

    # Gráfica 3: Lasso MSE val vs grado por alpha
    plt.figure()
    for a in ALPHAS:
        tmp = df_lasso[df_lasso["alpha"] == a].sort_values("degree")
        plt.plot(tmp["degree"], tmp["mse_val"], marker="o", label=f"a={a}")
    plt.title(fname + " Lasso: MSE val vs grado")
    plt.xlabel("grado"); plt.ylabel("MSE val")
    plt.legend()
    plt.show()

    # Gráfica 4: Lasso train vs val para alpha óptimo
    a_star = float(best_lasso["alpha"])
    tmp = df_lasso[df_lasso["alpha"] == a_star].sort_values("degree")
    plt.figure()
    plt.plot(tmp["degree"], tmp["mse_train"], marker="o", label="train")
    plt.plot(tmp["degree"], tmp["mse_val"], marker="o", label="val")
    plt.title(fname + f" Lasso (alpha={a_star}): train vs val")
    plt.xlabel("grado"); plt.ylabel("MSE")
    plt.legend()
    plt.show()

    # Mostrar solo el top 10
    print("Top 10 Ridge (menor MSE val):")
    display(df_ridge.sort_values("mse_val").head(10))
    print("Top 10 Lasso (menor MSE val):")
    display(df_lasso.sort_values("mse_val").head(10))


In [ ]:
df_best = pd.DataFrame(all_best)
display(df_best)

# Reflexiones

#### Resultados principales ≈ Mejor combinación por dataset

Dataset 1:
Ridge y Lasso coinciden en el grado 5. Ridge selecciona alpha = 1.0 (MSE_train ≈ 23.703, MSE_val ≈ 24.102) y Lasso alpha = 0.01 (MSE_train ≈ 23.703, MSE_val ≈ 24.102). Frente a OLS (MSE_val ≈ 24.000 con grado 5), la regularización no mejora, sino que empeora ligeramente la validación.

Dataset 2:
El mejor modelo pasa a grado 1 en ambos casos. Ridge elige alpha = 10.0 (MSE_val ≈ 24.049) y Lasso alpha = 0.01 (MSE_val ≈ 24.049). Comparado con OLS (grado 5 y MSE_val ≈ 24.000), el modelo regularizado es más simple pero no generaliza mejor (val es algo peor).

Dataset 3:
De nuevo el óptimo es grado 5. Ridge con alpha = 10.0 (MSE_val ≈ 24.101) y Lasso con alpha = 0.01 (MSE_val ≈ 24.102). Igual que en dataset 1, frente a OLS no hay mejora en validación.

Dataset 4:
El óptimo baja de grado 9 (OLS) a grado 7 con regularización. Ridge elige alpha = 1.0 (MSE_val ≈ 0.249) y Lasso alpha = 0.01 (MSE_val ≈ 0.250). Aunque el modelo se simplifica, el MSE de validación queda ligeramente peor que con OLS (≈0.248), por lo que aquí la regularización tampoco aporta una mejora clara.

Dataset 5:
Se mantiene grado 2, que ya era el óptimo en OLS. Ridge con alpha = 100.0 obtiene MSE_val ≈ 20159.221 (ligeramente mejor que OLS ≈ 20159.347), mientras que Lasso con alpha = 0.1 empeora un poco (MSE_val ≈ 20159.731). En este dataset los errores son muy altos, lo que sugiere que el resultado está dominado por ruido u outliers y la regularización tiene un efecto limitado.

Dataset 6:
Se mantiene grado 7, pero aquí sí se observa mejora real en generalización. Ridge con alpha = 0.1 reduce el MSE de validación ≈ 3.537 y Lasso con alpha = 0.01 lo baja a ≈ 3.535, frente a OLS ≈ 3.557. Este es el caso donde la regularización resulta más útil.

#### Interpretación sesgo–varianza con las curvas train vs val

El sesgo domina cuando el modelo es demasiado simple, basicamente tanto el error de entrenamiento como el de validación quedan altos ocurriendo un subajuste. La varianza domina cuando el modelo es demasiado complejo pues el error de entrenamiento baja mucho, pero el de validación se estanca o empeora produciendose el sobreajuste. La regularización introduce más sesgo al penalizar coeficientes con la intención de reducir la varianza y mejorar la generalización. De los resultados observados, ese beneficio se ve sobre todo en dataset 6, mientras que en los datasets 1–4 el modelo sin regularizar ya estaba en una zona estable y la penalización añade sesgo sin mejorar la validación. En dataset 5, el problema parece estar limitado por el ruido/outliers, así que la regularización apenas puede mejorar el MSE.

#### Justificación de la convergencia de Lasso

Lasso a veces no termina bien el cálculo y no converge, sobre todo cuando usamos polinomios de grado alto y un alpha muy pequeño. Esto pasa porque las nuevas variables del polinomio se parecen mucho entre sí y al modelo le cuesta encontrar la mejor solución. En nuestros resultados aparecen algunos avisos de este tipo en combinaciones extremas. Es normal y se puede reducir aumentando el número máximo de iteraciones o evitando grados muy altos con alphas muy pequeños. En cualquier caso, elegimos el modelo final usando el MSE de validación.

#### Conclusión global

Comparando con OLS (grados óptimos 5,5,5,9,2,7), con regularización los grados óptimos quedan:
Ridge: 5,1,5,7,2,7
Lasso: 5,1,5,7,2,7

La regularización es útil cuando reduce la varianza y mejora el MSE de validación (caso claro: dataset 6). Si OLS ya estaba cerca del mejor punto de generalización (datasets 1–4), la regularización puede no ayudar e incluso empeorar ligeramente la validación al introducir sesgo. Y si el dataset está dominado por ruido/outliers (dataset 5), la mejora esperable es pequeña.
